# Base path

In [0]:
# Define paths for Gold (aggregated) and Silver (cleaned) layers
gold_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold"
silver_path = "/Volumes/weather_catalog/weather_platform/weather_data/silver"

# Imports

In [0]:
# Import PySpark functions for aggregations and window operations
from pyspark.sql.functions import avg, max, min, sum, round, month, dense_rank, desc, lag, col
from pyspark.sql.window import Window

# Check silver layer

## Count of rows check

In [0]:
# Load Silver layer data
silver_df = spark.read.parquet(silver_path)

print(f"Rows: {silver_df.count()}")

## Schema check

In [0]:
# Verify Silver layer schema (date, city, source, temperature_c, precipitation_mm, year)
silver_df.printSchema()

# Agg - 1 Yearly City Summary
- Answers the following queries:
    - Average temperature per city per year
    - Highest average temperature city in a year
    - Lowest average temperature city in a year
    - Total rainfall by city per year
    - Rainiest city in a year
    - Driest city in a year
    - Temperature variation across years

In [0]:
# Aggregation 1: Calculate yearly statistics per city
# Groups by year and city, computes avg/max/min temperature and total/avg precipitation
yearly_city_summary = (
    silver_df
    .groupBy("year", "city")
    .agg(
        round(avg("temperature_c"), 2).alias("avg_temperature_c"),
        round(max("temperature_c"), 2).alias("max_temperature_c"),
        round(min("temperature_c"), 2).alias("min_temperature_c"),
        round(sum("precipitation_mm"), 2).alias("total_precipitation_mm"),
        round(avg("precipitation_mm"), 2).alias("avg_precipitation_mm")
    )
    .orderBy("year", "city")
)

display(yearly_city_summary)

## Validation

In [0]:
# Validate row count
print(f"Rows: {yearly_city_summary.count()}")

## Save - gold folder

### Path

In [0]:
# Define output path for yearly city summary aggregation
yearly_gold_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold/yearly_city_summary"

### Save into folder

In [0]:
# Save yearly city summary to Gold layer as Parquet
yearly_city_summary.write.mode("overwrite").parquet(yearly_gold_path)

print("Yearly City Summary saved successfully")

### Verification of saved dataset

In [0]:
# Verify saved dataset by reading it back and checking row count
gold_yearly_df = spark.read.parquet(yearly_gold_path)

print(f"Rows: {gold_yearly_df.count()}")

display(gold_yearly_df)

# Agg - 2 Monthly City Summary
- Answers the following queries:
    - Monthly temperature trends
    - Seasonal patterns
    - Monthly rainfall analysis
    - Weather trend analysis over time

In [0]:
# Aggregation 2: Calculate monthly statistics per city
# Extract month from date, then group by year, month, and city
monthly_city_summary = (
    silver_df
    .withColumn("month", month("date"))
)

# Aggregation
monthly_city_summary = (
    monthly_city_summary
    .groupBy("year", "month", "city")
    .agg(
        round(avg("temperature_c"), 2).alias("avg_temperature_c"),
        round(sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
    .orderBy("year", "month", "city")
)

display(monthly_city_summary)

## Validation

In [0]:
# Validate row count
print(f"Rows: {monthly_city_summary.count()}")

## Save - gold folder

### Path

In [0]:
# Define output path for monthly city summary aggregation
monthly_gold_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold/monthly_city_summary"

### Save into folder

In [0]:
# Save monthly city summary to Gold layer as Parquet
monthly_city_summary.write.mode("overwrite").parquet(monthly_gold_path)

print("Monthly City Summary saved successfully")

### Verification of saved dataset

In [0]:
# Verify saved monthly dataset by reading it back and checking row count
gold_monthly_df = spark.read.parquet(monthly_gold_path)

print(f"Rows: {gold_monthly_df.count()}")

display(gold_monthly_df)

# Agg - 3 Rainfall Ranking
- Answers the following queries:
    - Rainiest city in a year
    - Driest city in a year
    - Top 5 rainiest cities in a year
    - Top 10 rainfall rankings by year
    - Rainfall comparison between cities
    - Rainfall ranking across all cities
    - Highest rainfall recorded in a year
    - Lowest rainfall recorded in a year

In [0]:
# Aggregation 3: Rank cities by rainfall within each year
# Uses dense_rank() window function to assign rankings (1 = rainiest)
rainfall_ranking = (
    yearly_city_summary
    .withColumn(
        "rainfall_rank",
        dense_rank().over(
            Window.partitionBy("year")
                  .orderBy(desc("total_precipitation_mm"))
        )
    )
)

display(rainfall_ranking)

## Validation

In [0]:
# Validate row count
print(f"Rows: {rainfall_ranking.count()}")

## Check results

In [0]:
# Display top 20 records ordered by year and rank (shows rainiest cities first)
rainfall_ranking.orderBy(
    "year",
    "rainfall_rank"
).show(20, False)

## Save - gold folder

### Path

In [0]:
# Define output path for rainfall ranking aggregation
rainfall_ranking_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold/rainfall_ranking"

### Save into folder

In [0]:
# Save rainfall ranking to Gold layer as Parquet (overwrite mode)
rainfall_ranking.write.mode("overwrite").parquet(rainfall_ranking_path)

print("Rainfall Ranking saved successfully")

### Verification of saved dataset

In [0]:
# Verify saved rainfall ranking dataset by reading it back
gold_rainfall_df = spark.read.parquet(rainfall_ranking_path)

print(f"Rows: {gold_rainfall_df.count()}")

display(gold_rainfall_df)

# Agg - 4 Temperature Trend
- Answers the following queries:
    - Temperature variation across years for a city
    - Year-over-year temperature change
    - Cities with increasing temperature trends
    - Cities with decreasing temperature trends
    - Hottest trend city
    - Cooling trend city
    - Long-term climate trend analysis

In [0]:
# Aggregation 4: Calculate year-over-year temperature change per city
# Uses lag() window function to get previous year's temperature for comparison
temperature_trend = (
    yearly_city_summary
    .withColumn(
        "previous_year_temperature",
        lag("avg_temperature_c").over(
            Window.partitionBy("city")
                  .orderBy("year")
        )
    )
    .withColumn(
        "temperature_change",
        round(
            col("avg_temperature_c") - col("previous_year_temperature"), 2
        )
    )
)

display(temperature_trend)

## Validation

In [0]:
# Validate row count
print(f"Rows: {temperature_trend.count()}")

## Save - gold folder

### Path

In [0]:
# Define output path for temperature trend aggregation
temperature_trend_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold/temperature_trend"

### Save into folder

In [0]:
# Save temperature trend to Gold layer as Parquet
temperature_trend.write.mode("overwrite").parquet(temperature_trend_path)

print("Temperature Trend saved successfully")

### Validation of saved dataset

In [0]:
# Verify saved temperature trend dataset by reading it back
gold_temperature_df = spark.read.parquet(temperature_trend_path)

print(f"Rows: {gold_temperature_df.count()}")

display(gold_temperature_df)

# Agg - 5 Weather extremes Summary
- Answers the following queries:
    - Maximum/minimum temperatures
    - Maximum daily rainfall events
    - Extreme weather analysis

In [0]:
# Aggregation 5: Calculate weather extremes per city per year
# Finds highest/lowest temperatures and maximum daily rainfall
weather_extremes = (
    silver_df
    .groupBy("year", "city")
    .agg(
        round(max("temperature_c"), 2).alias("highest_temperature_c"),
        round(min("temperature_c"), 2).alias("lowest_temperature_c"),
        round(max("precipitation_mm"), 2).alias("highest_daily_rainfall_mm")
    )
    .orderBy("year", "city")
)

display(weather_extremes)

## Validation

In [0]:
# Validate row count
print(f"Rows: {weather_extremes.count()}")

## Save - gold folder

### Path

In [0]:
# Define output path for weather extremes aggregation
weather_extremes_path = "/Volumes/weather_catalog/weather_platform/weather_data/gold/weather_extremes"

### Save into folder

In [0]:
# Save weather extremes to Gold layer as Parquet
weather_extremes.write.mode("overwrite").parquet(weather_extremes_path)

print("Weather Extremes saved successfully")

### Validation of saved dataset

In [0]:
# Verify saved weather extremes dataset by reading it back
gold_weather_extremes_df = spark.read.parquet(weather_extremes_path)

print(f"Rows: {gold_weather_extremes_df.count()}")

display(gold_weather_extremes_df)